In [3]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torchinfo import summary
from torchviz import make_dot
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import torchvision.datasets as datasets
import sys
import os

In [ ]:
# Import custom module
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

%load_ext autoreload
%autoreload 2

from modules.Contrastive_Divergence_Minimisation import InteractionSufficientStatistics, ExponentialFamilyModel
# from modules.Annealed_Importance_Sampling import 

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [11]:
# set devise
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# prepare MNIST data
transform = transforms.Compose([
    transforms.ToTensor()
])

train_dataset = datasets.MNIST(root='./data', train=True, download=False, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=False, transform=transform)

train_loader = DataLoader(dataset=train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=64, shuffle=False)

Using device: cuda


In [12]:
# define 28*28 -> 50 -> 10 -> 5 -> 5 -> 5 -> 10 network
class MNISTNet(nn.Module):
    def __init__(self):
        super(MNISTNet, self).__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(784, 50)
        self.fc2 = nn.Linear(50, 10)
        self.fc3 = nn.Linear(10, 5)
        self.fc4 = nn.Linear(5, 5)
        self.fc5 = nn.Linear(5, 5)
        self.fc6 = nn.Linear(5, 10)

    def forward(self, x):
        x = self.flatten(x)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = torch.relu(self.fc3(x))
        x = torch.relu(self.fc4(x))
        x = torch.relu(self.fc5(x))
        x = self.fc6(x)
        return x
    
    def get_activations(self, x):
        x = self.flatten(x)
        act1 = torch.relu(self.fc1(x))
        act2 = torch.relu(self.fc2(act1))
        act3 = torch.relu(self.fc3(act2))
        act4 = torch.relu(self.fc4(act3))
        act5 = torch.relu(self.fc5(act4))
        act6 = self.fc6(act5)

        _, predicted = torch.max(act6, 1)

        return {
            'L1': act1.detach().cpu().numpy(),
            'L2': act2.detach().cpu().numpy(),
            'L3': act3.detach().cpu().numpy(),
            'L4': act4.detach().cpu().numpy(),
            'L5': act5.detach().cpu().numpy(),
            'L6': act6.detach().cpu().numpy(),
            'predicted': predicted.detach().cpu().numpy()
        }

In [13]:
# initialise model
model = MNISTNet().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
epochs = 100 

for epoch in range(epochs):
    # ==========================================
    # 1. learning phase
    # ==========================================
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {running_loss/len(train_loader):.4f}")
    
    # ==========================================
    # 2. obtain activation
    # ==========================================
    l1, l2, l3, l4, l5, l6 = [], [], [], [], [], []
    predictions = []
    
    model.eval() 
    with torch.no_grad(): 
        for images, labels in test_loader:
            images = images.to(device)
            
            activations = model.get_activations(images)
            l1.append(activations['L1'])
            l2.append(activations['L2'])
            l3.append(activations['L3'])
            l4.append(activations['L4'])
            l5.append(activations['L5'])
            l6.append(activations['L6'])
            predictions.append(activations['predicted'])
            
    l1 = np.vstack(l1)
    l2 = np.vstack(l2)
    l3 = np.vstack(l3)
    l4 = np.vstack(l4)
    l5 = np.vstack(l5)
    l6 = np.vstack(l6)
    predictions = np.concatenate(predictions)
    

Epoch [1/100], Loss: 1.6516
Epoch [2/100], Loss: 0.8473
Epoch [3/100], Loss: 0.6028
Epoch [4/100], Loss: 0.4645
Epoch [5/100], Loss: 0.3825
Epoch [6/100], Loss: 0.3277
Epoch [7/100], Loss: 0.2867
Epoch [8/100], Loss: 0.2553
Epoch [9/100], Loss: 0.2342
Epoch [10/100], Loss: 0.2148
Epoch [11/100], Loss: 0.1972
Epoch [12/100], Loss: 0.1839
Epoch [13/100], Loss: 0.1721
Epoch [14/100], Loss: 0.1610
Epoch [15/100], Loss: 0.1486
Epoch [16/100], Loss: 0.1410
Epoch [17/100], Loss: 0.1322
Epoch [18/100], Loss: 0.1241
Epoch [19/100], Loss: 0.1210
Epoch [20/100], Loss: 0.1141
Epoch [21/100], Loss: 0.1073
Epoch [22/100], Loss: 0.1014
Epoch [23/100], Loss: 0.0973
Epoch [24/100], Loss: 0.0925
Epoch [25/100], Loss: 0.0896
Epoch [26/100], Loss: 0.0862
Epoch [27/100], Loss: 0.0846
Epoch [28/100], Loss: 0.0796
Epoch [29/100], Loss: 0.0783
Epoch [30/100], Loss: 0.0744
Epoch [31/100], Loss: 0.0707
Epoch [32/100], Loss: 0.0698
Epoch [33/100], Loss: 0.0696
Epoch [34/100], Loss: 0.0661
Epoch [35/100], Loss: 0

In [16]:
with torch.no_grad():
    correct = 0
    total = 0
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"\nAccuracy on the test data: {100 * correct / total:.2f}%")


Accuracy on the test data: 95.47%
